In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

df = pd.read_csv('data/DIE_surgery_data_hall_oct2026.csv')

# 1. Variable sets
target = 'Any complication (ja/nein)'

ordinal_cols = [
    'ASA', 'ENZIAN P (0-3)', 'ENZIAN TL (0-3)', 'ENZIAN TR (0-3)',
    'ENZIAN OL (0-3)', 'ENZIAN OR (0-3)', 'ENZIAN A (0-3)',
    'ENZIAN BL (0-3)', 'ENZIAN BR (0-3)', 'ENZIAN C (0-3)'
]

binary_cols = [
    'ENZIAN FA (ja/nein)', 'ENZIAN FI (ja/nein)', 'ENZIAN FD (ja/nein)',
    'Voroperation Endometriose (ja/nein)'
]

continuous_cols = [
    'Präoperativer Hb (g/dl)', 'Δ Hb (prä-post Tag 1)', 'OP-Dauer (min)'
]

# 2. Harmonization & Cleaning
def binarize(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    return 1 if s in ['ja', 'yes', '1', '1.0', 'true'] else 0

df[target] = df[target].apply(binarize)
for c in binary_cols:
    df[c] = df[c].apply(binarize)

for c in ordinal_cols + continuous_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# 3. Bivariate Screening
records = []
all_vars = [
    ('Ordinal', col) for col in ordinal_cols
] + [
    ('Binary', col) for col in binary_cols
] + [
    ('Continuous', col) for col in continuous_cols
]

for var_type, col in all_vars:
    sub = df[[col, target]].dropna()
    g0 = sub[sub[target] == 0][col]
    g1 = sub[sub[target] == 1][col]
    
    n0, n1 = len(g0), len(g1)
    
    if var_type == 'Binary':
        # Fisher's Exact Test with 2x2 contingency matrix
        ct = pd.crosstab(sub[col], sub[target])
        if ct.shape == (2, 2):
            odds_ratio, p_val = stats.fisher_exact(ct)
        else:
            odds_ratio, p_val = np.nan, np.nan
        
        # Summary metrics: positive event proportions (%)
        p0 = (g0 == 1).mean() * 100 if n0 > 0 else np.nan
        p1 = (g1 == 1).mean() * 100 if n1 > 0 else np.nan
        summary_no_comp = f"{p0:.1f}% ({int(g0.sum())}/{n0})"
        summary_comp = f"{p1:.1f}% ({int(g1.sum())}/{n1})"
        metric_name = "OR"
        effect_val = odds_ratio

    elif var_type == 'Ordinal':
        # Mann-Whitney U test with rank-biserial correlation
        u_stat, p_val = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        r_rb = 1.0 - (2.0 * u_stat) / (n0 * n1) if (n0 * n1) > 0 else np.nan
        
        summary_no_comp = f"Mdn={g0.median():.0f} [IQR {g0.quantile(0.25):.0f}-{g0.quantile(0.75):.0f}]"
        summary_comp = f"Mdn={g1.median():.0f} [IQR {g1.quantile(0.25):.0f}-{g1.quantile(0.75):.0f}]"
        metric_name = "Rank-biserial r"
        effect_val = r_rb

    else:  # Continuous
        u_stat, p_val = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        r_rb = 1.0 - (2.0 * u_stat) / (n0 * n1) if (n0 * n1) > 0 else np.nan
        
        summary_no_comp = f"{g0.mean():.1f} ± {g0.std():.1f}"
        summary_comp = f"{g1.mean():.1f} ± {g1.std():.1f}"
        metric_name = "Rank-biserial r"
        effect_val = r_rb

    records.append({
        'Variable': col,
        'Type': var_type,
        'No Complication': summary_no_comp,
        'Complication (n=18)': summary_comp,
        'Effect Metric': metric_name,
        'Effect Size': effect_val,
        'p_raw': p_val
    })

res_df = pd.DataFrame(records)

# 4. Adjustment for Multiple Testing
valid_mask = res_df['p_raw'].notna()
res_df.loc[valid_mask, 'p_fdr_bh'] = multipletests(
    res_df.loc[valid_mask, 'p_raw'], method='fdr_bh'
)[1]
res_df.loc[valid_mask, 'p_bonferroni'] = multipletests(
    res_df.loc[valid_mask, 'p_raw'], method='bonferroni'
)[1]

res_df = res_df.sort_values('p_raw').reset_index(drop=True)

# 5. Format display
display_df = res_df.copy()
display_df['Effect Size'] = display_df['Effect Size'].map(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
for p_col in ['p_raw', 'p_fdr_bh', 'p_bonferroni']:
    display_df[p_col] = display_df[p_col].map(
        lambda x: "< 0.001" if x < 0.001 else f"{x:.3f}" if pd.notna(x) else "—"
    )

print(display_df.to_string(index=False))


                           Variable       Type No Complication Complication (n=18)   Effect Metric Effect Size p_raw p_fdr_bh p_bonferroni
            Präoperativer Hb (g/dl) Continuous      13.3 ± 1.0          12.9 ± 1.1 Rank-biserial r       -0.20 0.199    1.000        1.000
                     OP-Dauer (min) Continuous    219.4 ± 58.2        256.2 ± 87.9 Rank-biserial r        0.20 0.199    1.000        1.000
                    ENZIAN OL (0-3)    Ordinal Mdn=0 [IQR 0-2]     Mdn=2 [IQR 0-2] Rank-biserial r        0.17 0.262    1.000        1.000
                ENZIAN FD (ja/nein)     Binary     8.8% (5/57)         0.0% (0/18)              OR        0.00 0.329    1.000        1.000
                ENZIAN FI (ja/nein)     Binary   26.3% (15/57)        38.9% (7/18)              OR        1.78 0.376    1.000        1.000
                     ENZIAN A (0-3)    Ordinal Mdn=1 [IQR 0-3]     Mdn=2 [IQR 0-3] Rank-biserial r        0.12 0.409    1.000        1.000
                    ENZIAN 